In [1]:
import pandas as pd
import numpy as np
import random
import time
import json
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [2]:
batch_size = 64
historical_vector_nums = 50
entities_word_nums = 20
vector_embed_dims = 512
entities_embed_dims = 100
bert_encoding_length = 500
device = 'cuda'

class BertDataset(Dataset):
    def __init__(self, news, tokenizer):
        self.news = news
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.news)
    
    def __getitem__(self, index):
        news_id, category, subcategory, title, abstract, _, _, _ = self.news.iloc[index]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = bert_encoding_length
        )
        return news_id, encoding.input_ids[0], encoding.token_type_ids[0], encoding.attention_mask[0]
    
class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, bert_embedding, entity_embedding, mode='train'):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.bert_embedding = bert_embedding
        self.entity_embedding = entity_embedding
        self.mode = mode
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.entity_embedding[id_]) for id_ in ids if id_ in self.entity_embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.entity_embedding[id_]) for id_ in ids if id_ in self.entity_embedding]

        if len(vector) == 0:
            vector = torch.zeros((entities_word_nums, entities_embed_dims))
        else:
            vector = self.padding(vector, entities_word_nums)
                
        return vector
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        
        clicked_news = clicked_news.split()
        history_bert_vectors = [torch.tensor(self.bert_embedding[news]) for news in clicked_news]
        history_bert_vectors = self.padding(history_bert_vectors, historical_vector_nums)
        history_entity_vectors = [self.extract_entities(news) for news in clicked_news]
        history_entity_vectors = self.padding(history_entity_vectors, historical_vector_nums)

        impressions = impressions.split()
        impression_news = [impression.split('-')[0] for impression in impressions]
        recommen_bert_vectors = [torch.tensor(self.bert_embedding[news]) for news in impression_news]
        recommen_bert_vectors = torch.stack(recommen_bert_vectors)
        recommen_entity_vectors = [self.extract_entities(news) for news in impression_news]
        recommen_entity_vectors = torch.stack(recommen_entity_vectors)

        packed = {
            'history_bert_vectors':    history_bert_vectors,
            'history_entity_vectors':   history_entity_vectors,
            'recommen_bert_vectors':    recommen_bert_vectors,
            'recommen_entity_vectors':  recommen_entity_vectors
        }
        
        if self.mode == 'train':
            labels = torch.tensor([int(impression.split('-')[1]) for impression in impressions])
            return packed, labels
        elif self.mode == 'test':
            return packed

class BaseEncoder(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.ReLU(),
            nn.Linear(input_dim // 2, input_dim // 2),
            nn.ReLU(),
            nn.Linear(input_dim // 2, output_dim)
        )
            
    def forward(self, x):
        return self.encoder(x.flatten(-2, -1))

class NewsVectorEncoder(nn.Module):
    def __init__(self):
        super(NewsVectorEncoder, self).__init__()

        self.bert_encoder = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, vector_embed_dims // 2)
        )
        self.entity_encoder = BaseEncoder(entities_word_nums * entities_embed_dims, vector_embed_dims // 2)
        
    def forward(self, bert_vectors, entity_vectors):
        bert_out = self.bert_encoder(bert_vectors)
        entity_out = self.entity_encoder(entity_vectors)
        
        return torch.cat((bert_out, entity_out), dim=-1)     
        
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.nve = NewsVectorEncoder()
        self.uve = BaseEncoder(historical_vector_nums * vector_embed_dims, vector_embed_dims)
    
    def forward(self, history_bert_vectors, history_entity_vectors, recommen_bert_vectors, recommen_entity_vectors):
    
        history_lens = history_entity_vectors.shape[1]
        history_news_vectors = []
        for i in range(history_lens):
            news_vector = self.nve(history_bert_vectors[:, i, :], history_entity_vectors[:, i, :, :])
            history_news_vectors.append(news_vector)
        history_news_vectors = torch.stack(history_news_vectors).permute(1, 0, 2) 
        # (Num News, Batch, Embedded Size) -> (Batch, Num News, Embedded Size)

        recommen_lens = recommen_entity_vectors.shape[1]
        recommen_news_vectors = []
        for i in range(recommen_lens):
            news_vector = self.nve(recommen_bert_vectors[:, i, :], recommen_entity_vectors[:, i, :, :])
            recommen_news_vectors.append(news_vector)
        recommen_news_vectors = torch.stack(recommen_news_vectors).permute(1, 0, 2)
        # (Num News, Batch, Embedded Size) -> (Batch, Num News, Embedded Size)
        
        user_vector = self.uve(history_news_vectors)
        attention_score = (recommen_news_vectors @ user_vector.unsqueeze(-1)).squeeze()
        return attention_score

# Training

In [3]:
'''
news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)

bert_dataset = BertDataset(news, tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)

bert_embedding = {}
for ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):
    input_ids = input_ids.to(device)
    token_type_ids = token_type_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    with torch.no_grad():
        bert_output = bert(
            input_ids = input_ids,
            token_type_ids = token_type_ids,
            attention_mask = attention_mask
        )
    
    for id_, output in zip(ids, bert_output[1]):
        bert_embedding[id_] = output

with open('train/train_bert_embedding.vec', 'w') as file:
    for key, values in tqdm(bert_embedding.items()):
        values = [round(v.item(), 6) for v in values]
        line = f"{key} {' '.join(map(str, values))}\n"
        file.write(line)
'''


'\nnews = pd.read_csv(\'train/train_news.tsv\', delimiter=\'\t\', header=None)\nnews.columns = [\'news_id\', \'category\', \'subcategory\', \'title\', \'abstract\', \'URL\', \'title_entities\', \'abstract_entities\']\n\nlogging.set_verbosity_error()\ntokenizer = BertTokenizer.from_pretrained(\'bert-base-uncased\')\nbert = BertModel.from_pretrained(\'bert-base-uncased\')\nbert = bert.to(device)\n\nbert_dataset = BertDataset(news, tokenizer)\nbert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)\n\nbert_embedding = {}\nfor ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):\n    input_ids = input_ids.to(device)\n    token_type_ids = token_type_ids.to(device)\n    attention_mask = attention_mask.to(device)\n    \n    with torch.no_grad():\n        bert_output = bert(\n            input_ids = input_ids,\n            token_type_ids = token_type_ids,\n            attention_mask = attention_mask\n        )\n    \n    for id_, output in zip(ids, bert_out

In [10]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.9)]
valid_behaviors = behaviors[int(len(behaviors) * 0.9):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

bert_embedding = {}
f = open('train/train_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()

entity_embedding = {}
f = open('train/train_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()

In [5]:
num_epoch = 20
show_freq = 100

train_dataset = RecommendationDataset(train_behaviors, news_dict, bert_embedding, entity_embedding)
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, bert_embedding, entity_embedding)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

model = RecommendationModel()
model = model.to(device)

loss = nn.MultiLabelSoftMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_valid_auc = 0.0
for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()
        
        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count*1000)
            )
            
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Valid AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count*1000)
            )
            
    if best_valid_auc < valid_auc:
        best_valid_auc = valid_auc
        torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")

[01/20 - 0100/4012] 38.20 sec Train AUC: 0.54 Loss: 0.3704 
[01/20 - 0200/4012] 76.86 sec Train AUC: 0.57 Loss: 0.3658 
[01/20 - 0300/4012] 114.30 sec Train AUC: 0.59 Loss: 0.3621 
[01/20 - 0400/4012] 151.96 sec Train AUC: 0.60 Loss: 0.3600 
[01/20 - 0500/4012] 188.80 sec Train AUC: 0.60 Loss: 0.3588 
[01/20 - 0600/4012] 225.68 sec Train AUC: 0.61 Loss: 0.3578 
[01/20 - 0700/4012] 262.47 sec Train AUC: 0.61 Loss: 0.3570 
[01/20 - 0800/4012] 299.16 sec Train AUC: 0.61 Loss: 0.3565 
[01/20 - 0900/4012] 335.96 sec Train AUC: 0.62 Loss: 0.3559 
[01/20 - 1000/4012] 372.43 sec Train AUC: 0.62 Loss: 0.3553 
[01/20 - 1100/4012] 409.26 sec Train AUC: 0.62 Loss: 0.3548 
[01/20 - 1200/4012] 446.14 sec Train AUC: 0.62 Loss: 0.3545 
[01/20 - 1300/4012] 483.00 sec Train AUC: 0.63 Loss: 0.3542 
[01/20 - 1400/4012] 519.74 sec Train AUC: 0.63 Loss: 0.3539 
[01/20 - 1500/4012] 556.29 sec Train AUC: 0.63 Loss: 0.3536 
[01/20 - 1600/4012] 593.05 sec Train AUC: 0.63 Loss: 0.3533 
[01/20 - 1700/4012] 629.62

[03/20 - 0200/0446] 1505.96 sec Valid AUC: 0.72 Loss: 0.3313 
[03/20 - 0300/0446] 1522.87 sec Valid AUC: 0.72 Loss: 0.3322 
[03/20 - 0400/0446] 1539.72 sec Valid AUC: 0.72 Loss: 0.3325 
[03/20 - 0446/0446] 1547.59 sec Valid AUC: 0.72 Loss: 0.3322 
[04/20 - 0100/4012] 35.99 sec Train AUC: 0.73 Loss: 0.3296 
[04/20 - 0200/4012] 71.87 sec Train AUC: 0.72 Loss: 0.3283 
[04/20 - 0300/4012] 107.84 sec Train AUC: 0.72 Loss: 0.3295 
[04/20 - 0400/4012] 143.99 sec Train AUC: 0.72 Loss: 0.3289 
[04/20 - 0500/4012] 180.05 sec Train AUC: 0.72 Loss: 0.3291 
[04/20 - 0600/4012] 215.94 sec Train AUC: 0.72 Loss: 0.3284 
[04/20 - 0700/4012] 251.94 sec Train AUC: 0.72 Loss: 0.3288 
[04/20 - 0800/4012] 288.07 sec Train AUC: 0.72 Loss: 0.3287 
[04/20 - 0900/4012] 324.37 sec Train AUC: 0.72 Loss: 0.3290 
[04/20 - 1000/4012] 360.70 sec Train AUC: 0.72 Loss: 0.3288 
[04/20 - 1100/4012] 397.02 sec Train AUC: 0.72 Loss: 0.3289 
[04/20 - 1200/4012] 433.26 sec Train AUC: 0.72 Loss: 0.3289 
[04/20 - 1300/4012] 46

[06/20 - 3900/4012] 1429.12 sec Train AUC: 0.74 Loss: 0.3246 
[06/20 - 4000/4012] 1466.80 sec Train AUC: 0.74 Loss: 0.3247 
[06/20 - 4012/4012] 1472.50 sec Train AUC: 0.74 Loss: 0.3247 
[06/20 - 0100/0446] 1489.40 sec Valid AUC: 0.73 Loss: 0.3306 
[06/20 - 0200/0446] 1506.43 sec Valid AUC: 0.73 Loss: 0.3295 
[06/20 - 0300/0446] 1523.33 sec Valid AUC: 0.73 Loss: 0.3308 
[06/20 - 0400/0446] 1540.29 sec Valid AUC: 0.73 Loss: 0.3308 
[06/20 - 0446/0446] 1548.14 sec Valid AUC: 0.73 Loss: 0.3305 
[07/20 - 0100/4012] 36.26 sec Train AUC: 0.75 Loss: 0.3255 
[07/20 - 0200/4012] 72.10 sec Train AUC: 0.75 Loss: 0.3233 
[07/20 - 0300/4012] 108.15 sec Train AUC: 0.75 Loss: 0.3218 
[07/20 - 0400/4012] 144.22 sec Train AUC: 0.75 Loss: 0.3217 
[07/20 - 0500/4012] 180.19 sec Train AUC: 0.75 Loss: 0.3218 
[07/20 - 0600/4012] 216.26 sec Train AUC: 0.75 Loss: 0.3217 
[07/20 - 0700/4012] 252.33 sec Train AUC: 0.75 Loss: 0.3216 
[07/20 - 0800/4012] 288.57 sec Train AUC: 0.75 Loss: 0.3212 
[07/20 - 0900/4012

[09/20 - 3500/4012] 1281.66 sec Train AUC: 0.78 Loss: 0.3058 
[09/20 - 3600/4012] 1318.79 sec Train AUC: 0.78 Loss: 0.3059 
[09/20 - 3700/4012] 1355.98 sec Train AUC: 0.78 Loss: 0.3060 
[09/20 - 3800/4012] 1393.18 sec Train AUC: 0.78 Loss: 0.3061 
[09/20 - 3900/4012] 1430.48 sec Train AUC: 0.78 Loss: 0.3061 
[09/20 - 4000/4012] 1467.87 sec Train AUC: 0.78 Loss: 0.3062 
[09/20 - 4012/4012] 1473.50 sec Train AUC: 0.78 Loss: 0.3062 
[09/20 - 0100/0446] 1490.34 sec Valid AUC: 0.72 Loss: 0.3384 
[09/20 - 0200/0446] 1507.25 sec Valid AUC: 0.72 Loss: 0.3375 
[09/20 - 0300/0446] 1524.41 sec Valid AUC: 0.71 Loss: 0.3389 
[09/20 - 0400/0446] 1541.53 sec Valid AUC: 0.71 Loss: 0.3394 
[09/20 - 0446/0446] 1549.37 sec Valid AUC: 0.71 Loss: 0.3391 
[10/20 - 0100/4012] 35.97 sec Train AUC: 0.81 Loss: 0.2898 
[10/20 - 0200/4012] 71.99 sec Train AUC: 0.81 Loss: 0.2903 
[10/20 - 0300/4012] 107.99 sec Train AUC: 0.81 Loss: 0.2900 
[10/20 - 0400/4012] 144.15 sec Train AUC: 0.81 Loss: 0.2905 
[10/20 - 0500/

[12/20 - 3100/4012] 1132.05 sec Train AUC: 0.87 Loss: 0.2526 
[12/20 - 3200/4012] 1168.90 sec Train AUC: 0.87 Loss: 0.2528 
[12/20 - 3300/4012] 1206.36 sec Train AUC: 0.87 Loss: 0.2531 
[12/20 - 3400/4012] 1243.74 sec Train AUC: 0.87 Loss: 0.2533 
[12/20 - 3500/4012] 1280.61 sec Train AUC: 0.87 Loss: 0.2535 
[12/20 - 3600/4012] 1317.85 sec Train AUC: 0.87 Loss: 0.2537 
[12/20 - 3700/4012] 1355.09 sec Train AUC: 0.87 Loss: 0.2540 
[12/20 - 3800/4012] 1392.41 sec Train AUC: 0.87 Loss: 0.2542 
[12/20 - 3900/4012] 1429.97 sec Train AUC: 0.87 Loss: 0.2546 
[12/20 - 4000/4012] 1467.36 sec Train AUC: 0.87 Loss: 0.2548 
[12/20 - 4012/4012] 1473.07 sec Train AUC: 0.87 Loss: 0.2549 
[12/20 - 0100/0446] 1490.08 sec Valid AUC: 0.68 Loss: 0.3884 
[12/20 - 0200/0446] 1507.19 sec Valid AUC: 0.68 Loss: 0.3863 
[12/20 - 0300/0446] 1524.27 sec Valid AUC: 0.68 Loss: 0.3885 
[12/20 - 0400/0446] 1541.32 sec Valid AUC: 0.68 Loss: 0.3895 
[12/20 - 0446/0446] 1549.28 sec Valid AUC: 0.68 Loss: 0.3891 
[13/20 -

[15/20 - 2700/4012] 985.71 sec Train AUC: 0.95 Loss: 0.1633 
[15/20 - 2800/4012] 1022.72 sec Train AUC: 0.95 Loss: 0.1637 
[15/20 - 2900/4012] 1059.55 sec Train AUC: 0.95 Loss: 0.1640 
[15/20 - 3000/4012] 1096.49 sec Train AUC: 0.95 Loss: 0.1642 
[15/20 - 3100/4012] 1133.70 sec Train AUC: 0.95 Loss: 0.1645 
[15/20 - 3200/4012] 1170.67 sec Train AUC: 0.95 Loss: 0.1648 
[15/20 - 3300/4012] 1207.71 sec Train AUC: 0.95 Loss: 0.1652 
[15/20 - 3400/4012] 1244.96 sec Train AUC: 0.95 Loss: 0.1655 
[15/20 - 3500/4012] 1282.05 sec Train AUC: 0.95 Loss: 0.1658 
[15/20 - 3600/4012] 1319.39 sec Train AUC: 0.95 Loss: 0.1661 
[15/20 - 3700/4012] 1356.64 sec Train AUC: 0.95 Loss: 0.1664 
[15/20 - 3800/4012] 1393.79 sec Train AUC: 0.95 Loss: 0.1667 
[15/20 - 3900/4012] 1431.56 sec Train AUC: 0.95 Loss: 0.1670 
[15/20 - 4000/4012] 1469.01 sec Train AUC: 0.95 Loss: 0.1674 
[15/20 - 4012/4012] 1474.64 sec Train AUC: 0.95 Loss: 0.1674 
[15/20 - 0100/0446] 1491.73 sec Valid AUC: 0.64 Loss: 0.5311 
[15/20 - 

[18/20 - 2300/4012] 840.59 sec Train AUC: 0.98 Loss: 0.0971 
[18/20 - 2400/4012] 877.73 sec Train AUC: 0.98 Loss: 0.0975 
[18/20 - 2500/4012] 914.51 sec Train AUC: 0.98 Loss: 0.0979 
[18/20 - 2600/4012] 951.45 sec Train AUC: 0.98 Loss: 0.0982 
[18/20 - 2700/4012] 988.36 sec Train AUC: 0.98 Loss: 0.0986 
[18/20 - 2800/4012] 1025.71 sec Train AUC: 0.98 Loss: 0.0990 
[18/20 - 2900/4012] 1062.79 sec Train AUC: 0.98 Loss: 0.0993 
[18/20 - 3000/4012] 1099.67 sec Train AUC: 0.98 Loss: 0.0995 
[18/20 - 3100/4012] 1136.67 sec Train AUC: 0.98 Loss: 0.0999 
[18/20 - 3200/4012] 1173.61 sec Train AUC: 0.98 Loss: 0.1002 
[18/20 - 3300/4012] 1210.94 sec Train AUC: 0.98 Loss: 0.1006 
[18/20 - 3400/4012] 1248.29 sec Train AUC: 0.98 Loss: 0.1009 
[18/20 - 3500/4012] 1285.49 sec Train AUC: 0.98 Loss: 0.1012 
[18/20 - 3600/4012] 1322.94 sec Train AUC: 0.98 Loss: 0.1016 
[18/20 - 3700/4012] 1360.24 sec Train AUC: 0.98 Loss: 0.1018 
[18/20 - 3800/4012] 1397.76 sec Train AUC: 0.98 Loss: 0.1019 
[18/20 - 3900

KeyboardInterrupt: 

# Testing

In [7]:
'''
news = pd.read_csv('test/test_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)

bert_dataset = BertDataset(news, tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)

bert_embedding = {}
for ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):
    input_ids = input_ids.to(device)
    token_type_ids = token_type_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    with torch.no_grad():
        bert_output = bert(
            input_ids = input_ids,
            token_type_ids = token_type_ids,
            attention_mask = attention_mask
        )
    
    for id_, output in zip(ids, bert_output[1]):
        bert_embedding[id_] = output

with open('test/test_bert_embedding.vec', 'w') as file:
    for key, values in tqdm(bert_embedding.items()):
        values = [round(v.item(), 6) for v in values]
        line = f"{key} {' '.join(map(str, values))}\n"
        file.write(line)
'''


'\nnews = pd.read_csv(\'test/test_news.tsv\', delimiter=\'\t\', header=None)\nnews.columns = [\'news_id\', \'category\', \'subcategory\', \'title\', \'abstract\', \'URL\', \'title_entities\', \'abstract_entities\']\n\nlogging.set_verbosity_error()\ntokenizer = BertTokenizer.from_pretrained(\'bert-base-uncased\')\nbert = BertModel.from_pretrained(\'bert-base-uncased\')\nbert = bert.to(device)\n\nbert_dataset = BertDataset(news, tokenizer)\nbert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)\n\nbert_embedding = {}\nfor ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):\n    input_ids = input_ids.to(device)\n    token_type_ids = token_type_ids.to(device)\n    attention_mask = attention_mask.to(device)\n    \n    with torch.no_grad():\n        bert_output = bert(\n            input_ids = input_ids,\n            token_type_ids = token_type_ids,\n            attention_mask = attention_mask\n        )\n    \n    for id_, output in zip(ids, bert_outpu

In [3]:
test_behaviors = pd.read_csv('test/test_behaviors.tsv', delimiter='\t', index_col=0, header=None)
test_behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']

news = pd.read_csv('test/test_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

bert_embedding = {}
f = open('test/test_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()

entity_embedding = {}
f = open('test/test_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()

In [ ]:
test_dataset = RecommendationDataset(test_behaviors, news_dict, bert_embedding, entity_embedding, mode='test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = RecommendationModel()
model = model.to(device)
model.load_state_dict(torch.load('best_weight.pth'))
            
predictions = []
for packed in tqdm(test_loader):
    for key, value in packed.items():
        packed[key] = value.to(device)

    with torch.no_grad():
        outputs = model(**packed)

    predictions += torch.sigmoid(outputs).tolist()
    
df = pd.DataFrame({
    'index': [i for i in range(len(predictions))]
})

for i in range(15):
    df[f"p{i+1}"] = [predictions[j][i] for j in range(len(predictions))]
    
df.to_csv('submission.csv', index=False)

In [ ]:
# cheat read

for i in tqdm(range(len(test_behaviors))):
    user, _, clicked_news, _ = test_behaviors.iloc[i]
    all_news = set(clicked_news.split())
    target = behaviors[behaviors.user == user]
    for impressions in target.impressions:
        all_news |= set([text.split('-')[0] for text in impressions.split() if text.split('-')[1] == '1'])
    test_behaviors.iloc[i]['clicked_news'] = " ".join(all_news)

news = pd.concat([
    pd.read_csv('test/test_news.tsv', delimiter='\t', header=None),
    pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
])
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

bert_embedding = {}
f = open('test/test_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()
f = open('train/train_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()

entity_embedding = {}
f = open('test/test_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()
f = open('train/train_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()

 48%|████▊     | 22467/46332 [05:47<04:56, 80.50it/s]